<a href="https://colab.research.google.com/github/sravs-lab/chatbot/blob/main/chatbot_using_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langgraph langsmith


In [ ]:
!pip install langchain langchain_groq langchain_community


In [ ]:
from google.colab import userdata
from groq import Groq
groq_api_key=userdata.get('groq_api_key')
langsmith= userdata.get('langsmith_api_key')
print(langsmith)

In [ ]:
import os
os.environ['langchain_api_key']= langsmith
os.environ["langchain_tracing_v2"] = "true"
os.environ["langchain_project"] = "CourseLanggraph"

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile")

start building chatbot using langgraph

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import  StateGraph,START,END
from langgraph.graph.message import  add_messages

In [ ]:
class State(TypedDict):
  messages:Annotated[list, add_messages]
graph_builder = StateGraph(State)

In [ ]:
graph_builder

In [ ]:
def chatbot(state:State):
  return{"messages":llm.invoke(state["messages"])}

In [ ]:
graph_builder.add_node("chatbot",chatbot)

In [ ]:
graph_builder

In [ ]:
graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot",END)

In [ ]:
graph = graph_builder.compile()

In [ ]:
from IPython.display import  Image, display
try:
  display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
  pass

In [ ]:
while True:
  user_input= input("User: ")
  if user_input.lower() in ["quit", "q"]:
    print("Good Bye")
    break
  for event in graph.stream({'messages':("user",user_input)}):
    print(event.values())
    for value in event.values():
      print(value['messages'])
      print("Assistant:", value["messages"].content)

dict_values([{'messages': AIMessage(content='Hello. How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 36, 'total_tokens': 46, 'completion_time': 0.021891876, 'completion_tokens_details': None, 'prompt_time': 0.004976699, 'prompt_tokens_details': None, 'queue_time': 0.063498191, 'total_time': 0.026868575}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_d42c28f9ce', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e166f-912c-76b2-bfa8-c173e89e85cd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 10, 'total_tokens': 46})}])
content='Hello. How can I help you today?' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 36, 'total_tokens': 46, 'completion_time': 0.021891876, 'completion_tokens_details': None, 'prompt_time': 0.004976699, 'prom